In [1]:
import pandas as pd
import re
import sys
import numpy
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import LSTM, Dense, Dropout
from tensorflow.python.keras.callbacks import ModelCheckpoint
from tensorflow.python.keras.utils import np_utils
from more_itertools import windowed
from sklearn.preprocessing import OneHotEncoder
from collections import Counter
from tensorflow.python.keras.layers import Embedding
from tqdm import tqdm

In [2]:
def structure_lyrics(song_lyrics):
    
#     x = re.sub('.', '', song_lyrics)
    x = re.sub(',', '', song_lyrics)
    x = re.sub('\n\r\n', ' . ', x)
    x = re.sub('\r\n', ' , ', x)
    
    return x.lower()

In [3]:
# !wget https://github.com/gperakis/test_dataset/raw/master/stixoi_info_lyrics_all.zip
# !unzip stixoi_info_lyrics_all.zip

In [4]:
df = pd.read_csv('stixoi_info_lyrics_all.csv')
df.fillna('', inplace=True)

In [5]:
Counter(df['composers']).get('Φοίβος')

145

In [6]:
# df['singers'].value_counts()[40:100]

singers = [
    'Λευτέρης Πανταζής',
    'Χάρης Κωστόπουλος',
    'Μάκης Χριστοδουλόπουλος',
    'Βασίλης Τερλέγκας',
    'Βασίλης Καρράς',
    'Σταμάτης Γονίδης',
    'Αντύπας',
    'Πάνος Κιάμος',
    'Σώτης Βολάνης',
    'Ζαφείρης Μελάς',
    'Νίκος Οικονομόπουλος',
    'Γιάννης Πλούταρχος',
    'Κώστας Χαριτοδιπλωμένος',
    'Πασχάλης Τερζής',
    'Νίκος Μακρόπουλος',
    'Γιάννης Λεμπέσης']

In [7]:
# lyrics = df[df['lyricists']=='Φοίβος']['lyrics'].reset_index(drop=True)

# lyrics = df[df['singers'].isin(singers)]['lyrics'].reset_index(drop=True)

# lyrics = df['lyrics'].reset_index(drop=True)


In [8]:
lyrics = df[df['composers'] == 'Φοίβος']['lyrics'].reset_index(drop=True)

In [9]:
def structure_lyrics(song_lyrics):
    
    x = re.sub('.', '', song_lyrics)
    x = re.sub(',', '', song_lyrics)
    x = re.sub('\n\r\n', ' . ', x)
    x = re.sub('\r\n', ' , ', x)
    x = re.sub('\r', '', x)

    return x.lower()

structure_lyrics(lyrics[1])

'η μέρα εκείνη που σε πρωτοείδα , όλα τα άκουσα και όλα τα είδα , με έκανες την ζωή μου όλη να αλλάξω , χωρίς εσένα δεν μπορώ να υπάρξω. . δεν υπάρχει σαν και εσένα καμία , δημιούργησε ο θεός μόνο μια , πια μπορεί να δώσει αυτό το φιλί σου , ποιά να συγκριθεί μαζί σου. . γιατί καρδιά μου αξεπέραστη είσαι , στα δυο σου χέρια την ζωή μου κλείσε , για το μυαλό μου είσαι η μόνη σκέψη , όλα τα άλλα τα έχεις ανατρέψει. . δεν υπάρχει σαν και εσένα καμία , δημιούργησε ο θεός μόνο μια , πια μπορεί να δώσει αυτό το φιλί σου , ποιά να συγκριθεί μαζί σου.'

In [10]:
print(lyrics[1])

Η μέρα εκείνη που σε πρωτοείδα
όλα τα άκουσα και όλα τα είδα
με έκανες την ζωή μου όλη να αλλάξω
χωρίς εσένα δεν μπορώ να υπάρξω.

Δεν υπάρχει σαν και εσένα καμία
δημιούργησε ο Θεός μόνο μια
πια μπορεί να δώσει αυτό το φιλί σου
ποιά να συγκριθεί μαζί σου.

Γιατί καρδιά μου αξεπέραστη είσαι
στα δυο σου χέρια την ζωή μου κλείσε
για το μυαλό μου είσαι η μόνη σκέψη
όλα τα άλλα τα έχεις ανατρέψει.

Δεν υπάρχει σαν και εσένα καμία
δημιούργησε ο Θεός μόνο μια
πια μπορεί να δώσει αυτό το φιλί σου
ποιά να συγκριθεί μαζί σου.


In [11]:
from tensorflow.python.keras.preprocessing.text import Tokenizer

In [12]:
max_features = 3_000

tokenizer = Tokenizer(num_words=max_features, 
                      filters='!"#$%&()*+-/:;<=>?@[\\]^_`{|}~\t\n\r', 
                      lower=True, 
                      split=' ',
                      char_level=False,
                      oov_token='<OOV>',
                      document_count=1)

In [13]:
tokenizer.fit_on_texts(lyrics)

In [14]:
tokenizer.word_index
len(tokenizer.index_word)

3838

In [15]:
max_features = min([max_features, len(tokenizer.word_index)])
max_features

3000

In [16]:
text = structure_lyrics(lyrics[1])
text

'η μέρα εκείνη που σε πρωτοείδα , όλα τα άκουσα και όλα τα είδα , με έκανες την ζωή μου όλη να αλλάξω , χωρίς εσένα δεν μπορώ να υπάρξω. . δεν υπάρχει σαν και εσένα καμία , δημιούργησε ο θεός μόνο μια , πια μπορεί να δώσει αυτό το φιλί σου , ποιά να συγκριθεί μαζί σου. . γιατί καρδιά μου αξεπέραστη είσαι , στα δυο σου χέρια την ζωή μου κλείσε , για το μυαλό μου είσαι η μόνη σκέψη , όλα τα άλλα τα έχεις ανατρέψει. . δεν υπάρχει σαν και εσένα καμία , δημιούργησε ο θεός μόνο μια , πια μπορεί να δώσει αυτό το φιλί σου , ποιά να συγκριθεί μαζί σου.'

In [17]:
lyrics_indices_raw = tokenizer.texts_to_sequences(lyrics)

In [18]:
sentences, next_words = list(), list()

max_len = 5
step = 1

for song_indexes in tqdm(lyrics_indices_raw):    
    for w in windowed(seq=song_indexes,
                      n=(max_len + 1),
                      step=step):

        before = w[:-1]
        targ_word = w[-1]
        
        if targ_word:
            sentences.append(before)
            next_words.append(targ_word)

100%|██████████| 145/145 [00:00<00:00, 7464.82it/s]


In [19]:
print(f'Number of extracted Sequences: {len(sentences)}')
print(f'Number of extracted next words: {len(next_words)}')
print(f'Length of Char2Index: {len(tokenizer.word_index)}')
print(f'Length of Index2Char: {len(tokenizer.index_word)}')

Number of extracted Sequences: 19459
Number of extracted next words: 19459
Length of Char2Index: 3838
Length of Index2Char: 3838


In [20]:
import numpy as np

In [21]:
X = np.array(sentences)

In [22]:
y = np_utils.to_categorical(next_words,
                            num_classes= max_features + 1)

In [23]:
print(X.shape)
print(y.shape)

(19459, 5)
(19459, 3001)


In [24]:
from tensorflow.python.keras.models import Model
from tensorflow.python.keras.layers import Dense, Embedding, LSTM, Input, Concatenate, Bidirectional, concatenate

In [25]:
tokenizer

In [26]:
max_feats = len(tokenizer.word_index)
emb_dimensions = 300
n_outputs = y.shape[1]

# max_len =  yparxei idi pio panw


# this is the placeholder tensor for the input sequences
sequence = Input(shape=(max_len,), dtype='int32')

# this embedding layer will transform the sequences of integers into vectors of size 128
emb_layer = Embedding(max_feats,
                      emb_dimensions,
                      input_length=max_len)

embedded = emb_layer(sequence)


lstm = Bidirectional(LSTM(64, return_sequences=True, recurrent_dropout=0.2, dropout=0.2))(embedded)
lstm2 = Bidirectional(LSTM(64, return_sequences=False, recurrent_dropout=0.2, dropout=0.2))(lstm)

output = Dense(n_outputs, activation='softmax')(lstm2)

model = Model(inputs=[sequence], outputs=[output])

# try using different optimizers and different optimizer configs
model.compile('adam',
              'categorical_crossentropy',
              metrics=['accuracy'])

print(model.summary())

Model: "functional_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 5)]               0         
_________________________________________________________________
embedding (Embedding)        (None, 5, 300)            1151400   
_________________________________________________________________
bidirectional (Bidirectional (None, 5, 128)            186880    
_________________________________________________________________
bidirectional_1 (Bidirection (None, 128)               98816     
_________________________________________________________________
dense (Dense)                (None, 3001)              387129    
Total params: 1,824,225
Trainable params: 1,824,225
Non-trainable params: 0
_________________________________________________________________
None


In [30]:
from keras.callbacks import EarlyStopping

es = EarlyStopping(patience=5,
                   monitor='loss',
                   restore_best_weights=True)

In [ ]:
model.fit(X,
          y, 
          epochs=200,
          batch_size=1024,
          verbose=1,
          callbacks=[es],
          validation_split=0.1)

Epoch 1/200
18/18 [==============================] - 4s 220ms/step - loss: 6.2694 - accuracy: 0.0395 - val_loss: 6.4065 - val_accuracy: 0.0637
Epoch 2/200
18/18 [==============================] - 4s 217ms/step - loss: 6.2460 - accuracy: 0.0387 - val_loss: 6.4412 - val_accuracy: 0.0411
Epoch 3/200
18/18 [==============================] - 4s 214ms/step - loss: 6.2140 - accuracy: 0.0382 - val_loss: 6.4796 - val_accuracy: 0.0637
Epoch 4/200
18/18 [==============================] - 4s 214ms/step - loss: 6.1570 - accuracy: 0.0405 - val_loss: 6.5073 - val_accuracy: 0.0555
Epoch 5/200
18/18 [==============================] - 4s 219ms/step - loss: 6.0627 - accuracy: 0.0447 - val_loss: 6.5591 - val_accuracy: 0.0550
Epoch 6/200
18/18 [==============================] - 4s 217ms/step - loss: 5.9981 - accuracy: 0.0532 - val_loss: 6.5794 - val_accuracy: 0.0627
Epoch 7/200
18/18 [==============================] - 4s 219ms/step - loss: 5.9363 - accuracy: 0.0582 - val_loss: 6.6335 - val_accuracy: 0.0596

In [ ]:
model.save_weights('diridata.h5')